# Glance: Attribute-Aware Fashion and Context Retrieval

This sequential notebook uses the existing modular **CLIP + ChromaDB** implementation. It improves on vanilla CLIP by extracting query attributes, retrieving a broad vector shortlist, and re-ranking it with transparent fashion and scene metadata.

## 1. Setup and reproducibility

Run from the project root. This notebook does not clone a placeholder repository, request an unnecessary token, or delete an existing index. Re-indexing is safe because image records are upserted by filename.

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

# Colab opens a notebook in /content, whereas local Jupyter usually opens it
# in the repository. Bootstrap the real project only when config.py is absent.
IS_COLAB = "google.colab" in sys.modules
REPOSITORY_URL = os.environ.get(
    "GLANCE_REPOSITORY_URL",
    "https://github.com/kwazzy-coder/Glance-Fashion-Retrieval.git",
)
REPOSITORY_DIR = Path("/content/Glance-Fashion-Retrieval") if IS_COLAB else Path.cwd()

if not Path("config.py").is_file():
    if IS_COLAB:
        if not (REPOSITORY_DIR / ".git").is_dir():
            subprocess.check_call(["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_DIR)])
        os.chdir(REPOSITORY_DIR)
    else:
        raise RuntimeError("Run this notebook from the Glance project root (the folder containing config.py).")

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.py").is_file():
    raise RuntimeError(f"Glance project was not found at {PROJECT_ROOT}.")

required = ("torch", "open_clip", "chromadb", "PIL", "pandas", "matplotlib")
missing = [package for package in required if importlib.util.find_spec(package) is None]
if missing:
    print("Installing missing dependencies:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
import config

print(f"Runtime: {'Google Colab' if IS_COLAB else 'local Jupyter'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {config.DEVICE}")
print(f"Image directory: {config.IMAGE_DIR}")
print(f"ChromaDB directory: {config.CHROMA_PERSIST_DIR}")


In [ ]:
# CLIP is loaded lazily by the modular pipelines. A local checkpoint is used
# when available; otherwise open_clip resolves the configured OpenAI CLIP weights.
checkpoint = config.resolve_clip_checkpoint()
if checkpoint:
    config.CLIP_CHECKPOINT_PATH = checkpoint
    config.OPEN_CLIP_PRETRAINED = checkpoint
    print(f"Using local CLIP checkpoint: {checkpoint}")
else:
    print("No local checkpoint found; open_clip will resolve the configured weights.")
print(f"Model: {config.OPEN_CLIP_MODEL} / {config.OPEN_CLIP_PRETRAINED}")

## 2. Dataset and indexing

The indexer uses CLIP both to create zero-shot fashion/context captions and to encode the image. It persists fused vectors plus clothing, color, accessory, environment, and style metadata in ChromaDB. Batching and the `VectorStore` interface keep this stage ready for a larger backend.

In [ ]:
image_paths = sorted(path for path in config.IMAGE_DIR.iterdir()
                     if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
if not image_paths:
    print("No images found; downloading the configured Fashionpedia validation subset.")
    subprocess.check_call([sys.executable, "download_dataset.py", "--count", str(config.MAX_IMAGES)])
    image_paths = sorted(path for path in config.IMAGE_DIR.iterdir()
                         if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"})
print(f"Images available: {len(image_paths)}")
if not image_paths:
    raise RuntimeError("Dataset download produced no images; check the network and rerun this cell.")

In [ ]:
# Guard against a persisted index created by a different CLIP model.
# ChromaDB fixes a collection's vector width on first insert, so a 768-d index
# cannot accept this ViT-B/32 pipeline's 512-d vectors. Recreate only when the
# stored width differs; compatible collections remain intact for resumable runs.
import chromadb

expected_dimension = config.EMBEDDING_DIM
chroma_client = chromadb.PersistentClient(path=str(config.CHROMA_PERSIST_DIR))
chroma_collection = chroma_client.get_or_create_collection(
    name=config.CHROMA_COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

stored_dimension = None
if chroma_collection.count() > 0:
    stored_embeddings = chroma_collection.get(limit=1, include=["embeddings"]).get("embeddings")
    if stored_embeddings is not None and len(stored_embeddings) > 0:
        stored_dimension = len(stored_embeddings[0])

if stored_dimension is not None and stored_dimension != expected_dimension:
    print(f"Recreating stale {stored_dimension}-d ChromaDB index for {expected_dimension}-d CLIP vectors.")
    chroma_client.delete_collection(config.CHROMA_COLLECTION_NAME)
else:
    print(f"ChromaDB is compatible ({stored_dimension or 'new'} ? {expected_dimension} dimensions).")


In [ ]:
import logging
from indexer.index_pipeline import IndexPipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
indexer = IndexPipeline()
index_summary = indexer.index_directory(
    str(config.IMAGE_DIR), max_images=config.MAX_IMAGES, batch_size=config.BATCH_SIZE
)
print("Index summary:", index_summary)

In [ ]:
from indexer.vector_store import VectorStore
store = VectorStore()
print(f"Indexed records: {store.get_collection_size()}")
if store.get_collection_size() == 0:
    raise RuntimeError("The vector index is empty; resolve indexing errors before retrieval.")

## 3. Inspect indexed metadata

Captions and metadata are the reranker's evidence. Review a few records before evaluating: generic captions are a common cause of weak fashion and environment retrieval.

In [ ]:
sample = store.collection.peek(limit=min(5, store.get_collection_size()))
for image_id, metadata in zip(sample["ids"], sample["metadatas"]):
    print(f"\n{image_id}")
    print(" caption:", metadata.get("caption", ""))
    print(" attributes:", {key: metadata.get(key, "") for key in
          ("clothing_types", "colors", "accessories", "environment", "style")})
    image_path = Path(metadata.get("image_path", ""))
    if image_path.is_file():
        plt.figure(figsize=(3, 3))
        plt.imshow(Image.open(image_path).convert("RGB"))
        plt.title(image_id, fontsize=9)
        plt.axis("off")
        plt.show()

## 4. Retrieval architecture

```mermaid
flowchart TD
    A["Query"] --> B["Query Attribute Extraction"]
    B --> C["CLIP Text Embedding"]
    C --> D["ChromaDB Vector Search (Top-N)"]
    D --> E["Attribute-aware Reranking"]
    B --> E
    E --> F["Final Top-K Results"]
```

Stage 1 favors recall. Stage 2 weights fashion information more heavily than background: compositional color?garment pairs (40% of attribute score), garment type (25%), color (20%), accessories (8%), environment (5%), and style (2%). Final score: `0.65 ? normalized vector relevance + 0.35 ? attribute score`.

In [ ]:
from retriever.retrieve_pipeline import RetrievePipeline
retriever = RetrievePipeline()
decomposer = retriever._decomposer  # Diagnostic read-only access for notebook display.

In [ ]:
def compact_match_explanation(matched_attributes: dict) -> str:
    """Turn structured reranking evidence into a concise explanation."""
    parts = []
    for field, detail in matched_attributes.items():
        if not isinstance(detail, dict):
            continue
        hits, score = detail.get("hits", []), float(detail.get("score", 0.0))
        if hits:
            parts.append(f"{field}: {', '.join(map(str, hits))}")
        elif score > 0:
            parts.append(f"{field}: {score:.2f}")
    return " ? ".join(parts) or "semantic match only"


def display_results(query: str, results: list[dict], attributes: dict) -> None:
    """Show extracted query attributes, transparent scores, and image results."""
    display(pd.DataFrame([{
        "clothing items": ", ".join(f"{i.get('color') or 'any'} {i['type']}" for i in attributes.get("clothing_items", [])) or "none",
        "accessories": ", ".join(attributes.get("accessories", [])) or "none",
        "environment": attributes.get("environment") or "none",
        "style": attributes.get("style") or "none",
        "expanded search text": attributes.get("enhanced_query", query),
    }]))
    rows = [{"rank": rank, "image": Path(r["image_path"]).name,
             "final score": r["final_score"], "vector relevance": r["vector_similarity"],
             "attribute match": r["attribute_score"],
             "why selected": compact_match_explanation(r["matched_attributes"])}
            for rank, r in enumerate(results, 1)]
    display(pd.DataFrame(rows))
    figure, axes = plt.subplots(1, max(1, len(results)), figsize=(4 * max(1, len(results)), 5))
    axes = [axes] if len(results) == 1 else axes
    for axis, result in zip(axes, results):
        path = Path(result["image_path"])
        if path.is_file():
            axis.imshow(Image.open(path).convert("RGB"))
        axis.set_title(f"Rank {result['image_id']}\nfinal={result['final_score']:.3f} | vector={result['vector_similarity']:.3f} | attr={result['attribute_score']:.3f}", fontsize=8)
        axis.axis("off")
    plt.suptitle(query, fontweight="bold")
    plt.tight_layout()
    plt.show()

## 5. Benchmark retrieval and compositional checks

The final two queries deliberately reverse garment/color bindings. The reranker checks whether each color occurs next to its garment in the caption, reducing the bag-of-words ambiguity of vanilla embedding similarity.

In [ ]:
BENCHMARK_QUERIES = [
    "A person in a bright yellow raincoat.",
    "Professional business attire inside a modern office.",
    "Someone wearing a blue shirt sitting on a park bench.",
    "Casual weekend outfit for a city walk.",
    "A red tie and a white shirt in a formal setting.",
    "red shirt with blue pants",
    "blue shirt with red pants",
]
benchmark_results = {}
for query in BENCHMARK_QUERIES:
    attributes = decomposer.decompose(query)
    results = retriever.retrieve(query, top_k=5)
    benchmark_results[query] = results
    print(f"\nQuery: {query}")
    display_results(query, results, attributes)

## 6. Quantitative evaluation against vanilla CLIP

Label the small validation set once after indexing. Each `relevant_image_ids` list must contain every visually judged relevant image for its query. The next cell compares CLIP's same Top-N candidates before reranking with the final attribute-aware Top-K, then displays exact Precision@K and Recall@K.

In [ ]:
# Replace the empty lists after visually judging the result contact sheets.
# Keep these labels with the experiment to make the metrics reproducible.
MANUAL_VALIDATION_SET = [
    {"query": "red shirt with blue pants", "relevant_image_ids": []},
    {"query": "blue shirt with red pants", "relevant_image_ids": []},
    {"query": "formal office outfit with a red tie", "relevant_image_ids": []},
    {"query": "casual outfit in a park", "relevant_image_ids": []},
    {"query": "yellow raincoat outdoors", "relevant_image_ids": []},
]
if not any(example["relevant_image_ids"] for example in MANUAL_VALIDATION_SET):
    print("Label relevant image IDs in MANUAL_VALIDATION_SET, then rerun this and the next cell.")
    print("Copy IDs from the result tables above.")

In [ ]:
def precision_recall_at_k(retrieved_ids: list[str], relevant_ids: set[str], k: int) -> tuple[float, float]:
    """Compute exact metrics from manually judged relevance IDs."""
    hits = sum(image_id in relevant_ids for image_id in retrieved_ids[:k])
    return hits / k, hits / len(relevant_ids)


def evaluate_pipeline(validation_set: list[dict], k: int = 5) -> pd.DataFrame:
    """Compare vanilla stage-1 CLIP ranking with attribute-aware reranking."""
    rows = []
    for example in validation_set:
        relevant_ids = set(example["relevant_image_ids"])
        if not relevant_ids:
            continue
        query = example["query"]
        attributes = decomposer.decompose(query)
        vanilla = retriever._search_engine.search(query, attributes, top_k=config.TOP_K_INITIAL)
        reranked = retriever._reranker.rerank(vanilla, attributes, top_k=k)
        vanilla_p, vanilla_r = precision_recall_at_k([r["image_id"] for r in vanilla], relevant_ids, k)
        reranked_p, reranked_r = precision_recall_at_k([r["image_id"] for r in reranked], relevant_ids, k)
        rows += [
            {"query": query, "method": "vanilla CLIP", f"Precision@{k}": vanilla_p, f"Recall@{k}": vanilla_r},
            {"query": query, "method": "attribute-aware CLIP", f"Precision@{k}": reranked_p, f"Recall@{k}": reranked_r},
        ]
    return pd.DataFrame(rows)


evaluation_table = evaluate_pipeline(MANUAL_VALIDATION_SET, k=5)
if evaluation_table.empty:
    print("No metrics yet: add manually judged relevant image IDs in the preceding cell.")
else:
    display(evaluation_table.style.format({"Precision@5": "{:.3f}", "Recall@5": "{:.3f}"}))
    display(evaluation_table.groupby("method", as_index=False)[["Precision@5", "Recall@5"]].mean().style.format({"Precision@5": "{:.3f}", "Recall@5": "{:.3f}"}))

## 7. Interactive retrieval

In [ ]:
custom_query = "A woman in a red dress at a formal event"  # Change this query.
custom_attributes = decomposer.decompose(custom_query)
custom_results = retriever.retrieve(custom_query, top_k=5)
display_results(custom_query, custom_results, custom_attributes)

## 8. Scalability notes

`SearchEngine` depends on the `VectorStore` interface, not notebook state. At millions of images, stream batches from object storage, precompute/version captions and attributes, shard ChromaDB or replace only `VectorStore` with a managed ANN backend, and retain the bounded Top-N ? rerank boundary.

In [ ]:
# Optional portable copy; it never alters the live index.
archive_path = shutil.make_archive("chromadb_export", "zip", config.CHROMA_PERSIST_DIR)
print(f"Created {archive_path} ({Path(archive_path).stat().st_size / 1e6:.1f} MB)")